In [1]:
from datasets import load_dataset
import pandas as pd

/Users/administrador/Desktop/2026_Hack4Edu/Hack4Edu_2026/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_english_math = load_dataset('masakhane/afrimgsm', 'eng') 
data_yor_math = load_dataset('masakhane/afrimgsm', "yor") 

In [3]:
data_english_math, data_yor_math

(DatasetDict({
     train: Dataset({
         features: ['question', 'answer', 'answer_number', 'equation_solution'],
         num_rows: 8
     })
     test: Dataset({
         features: ['question', 'answer', 'answer_number', 'equation_solution'],
         num_rows: 250
     })
 }),
 DatasetDict({
     train: Dataset({
         features: ['question', 'answer', 'answer_number', 'equation_solution'],
         num_rows: 8
     })
     test: Dataset({
         features: ['question', 'answer', 'answer_number', 'equation_solution'],
         num_rows: 250
     })
 }))

In [4]:
data_yor_math["train"]["equation_solution"]

Column(['5 + 6 = 11', '4 * 5 = 20. 9 + 20 = 29', '32 + 42 = 74. 74 - 35 = 39', '5 + 2 = 7. 7 + 2 = 9', '58 - 23 = 35. 35 - 2 = 33', ...])

In [5]:
data_yor_math["train"]["answer"][0]

'Ìdáhùn lẹ́sẹsẹ: Roger bẹ̀rẹ̀ pẹ̀lú bọ́ọ́lù 5. Agolo 2 bọ́ọ́lù aláfajọ̀gbá kọ̀ọ̀kan 3 jẹ́ bọ́ọ́lù aláfajọ̀gbá 6. 5 + 6 = 11. Ìdáhùn náà ni 11.'

In [6]:
rag_rows = []

for split in ["train", "test"]:
    number_rows = len(data_english_math[split]["question"])
    
    for index in range(number_rows):
        q_yor = data_yor_math[split]["question"][index]
        q_english = data_english_math[split]["question"][index]
        
        ans_yor = data_yor_math[split]["answer"][index]
        
        ans_num = data_yor_math[split]["answer_number"][index]
        eq_english = data_yor_math[split]["equation_solution"][index]
        
        
        context = f"{q_yor} | {q_english}"
        
        
        if ans_yor is not None and eq_english is not None:
            answer = (
                f"[Verified Translation]: {q_english}\n"
                f"[Yorùbá Step-by-Step]: {ans_yor}\n"
                f"[Math Equation]: {eq_english}\n"
                f"[Exact Final Answer]: {ans_num}"
            )
        
        else:
            answer = (
                f"[Verified Parallel Math Problem (Yorùbá -> English)]:\n"
                f"- Yorùbá Text: {q_yor}\n"
                f"- Exact English Meaning: {q_english}\n"
                f"- Verified Numerical Result: {ans_num}"
            )
            
        rag_rows.append({
            "language": "yor",
            "source_dataset": f"afrimgsm_{split}",
            "search_context": context,
            "pedagogical_answer": answer
        })

df_african = pd.DataFrame(rag_rows)

In [7]:
df_african["pedagogical_answer"].values

<ArrowStringArray>
[                                                                                                                                                                                                                                                                                                                                                                                                                                                                       '[Verified Translation]: Question: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?\n[Yorùbá Step-by-Step]: Ìdáhùn lẹ́sẹsẹ: Roger bẹ̀rẹ̀ pẹ̀lú bọ́ọ́lù 5. Agolo 2 bọ́ọ́lù aláfajọ̀gbá kọ̀ọ̀kan 3 jẹ́ bọ́ọ́lù aláfajọ̀gbá 6. 5 + 6 = 11. Ìdáhùn náà ni 11.\n[Math Equation]: 5 + 6 = 11\n[Exact Final Answer]: 11',
                                                                                                                                       

In [8]:
print(df_african.tail(3)["pedagogical_answer"].values[0])

[Verified Parallel Math Problem (Yorùbá -> English)]:
- Yorùbá Text: Isabelle ra àwọn ohun èlò ayẹyẹ fún ayẹyẹ ọjọ́ ìbí àbúrò rẹ̀ obìnrin. Ó ná $12 lórí àwọn èròjà fún àkàrà òyìnbó náà, $43 lórí ẹ̀bùn ọjọ́ ìbí, $15 lórí àwọn ohun ọ̀ṣọ́, $4 lórí ìfìwépeni, àti $22 lórí àwọn àpò goodie fún àwọn àlejò ayẹyẹ. Ó pín iye náà ní ọ̀nà mẹ́ta pẹ̀lú àwọn òbí rẹ̀ méjèèjì. Dọ́là mélòó ni ẹnì kọ̀ọ̀kan san?
- Exact English Meaning: Isabelle bought party supplies for her little sister’s birthday party. She spent $12 on ingredients for the cake, $43 on birthday presents, $15 on decorations, $4 on invitations, and $22 on goodie bags for the party guests. She split the cost evenly three ways with her two parents. How many dollars did each person pay?
- Verified Numerical Result: 32


In [9]:
df_african

,language,source_dataset,search_context,pedagogical_answer
0,yor,afrimgsm_train,Ìbéèrè: Roger ní bọ́ọ́lù aláfajọ̀ 5. Ó ra agol...,[Verified Translation]: Question: Roger has 5 ...
1,yor,afrimgsm_train,Ìbéèrè: Kọ̀mpútà mẹ́sàn-án ni ó wà ní yàrá ojú...,[Verified Translation]: Question: There were n...
2,yor,afrimgsm_train,Ìbéèrè: Leah ní ṣokolétì 32 bẹ́ẹ̀ sì ni arábìn...,[Verified Translation]: Question: Leah had 32 ...
3,yor,afrimgsm_train,Ìbéèrè: Shawn ní ohun ìṣeré márùn-ún. Fún ọdún...,[Verified Translation]: Question: Shawn has fi...
4,yor,afrimgsm_train,Ìbéèrè: Michael ní bọ́ọ́lù aláfigigbá sínú ihò...,[Verified Translation]: Question: Michael had ...
...,...,...,...,...
253,yor,afrimgsm_test,Àwọn ìgbín mẹ́rin ló wà nínú ilé-ẹja kan àti ì...,[Verified Parallel Math Problem (Yorùbá -> Eng...
254,yor,afrimgsm_test,Oníṣẹ́-ọnà jeans máa ń san $250 fún un ní wákà...,[Verified Parallel Math Problem (Yorùbá -> Eng...
255,yor,afrimgsm_test,Isabelle ra àwọn ohun èlò ayẹyẹ fún ayẹyẹ ọjọ́...,[Verified Parallel Math Problem (Yorùbá -> Eng...
256,yor,afrimgsm_test,Alásè kan ra àpò àlùbọ́sà mẹ́rin. Àpò kọ̀ọ̀kan...,[Verified Parallel Math Problem (Yorùbá -> Eng...
